In [62]:
import pandas as pd
import numpy as np
import re

In [63]:
filepath = 'b17f8_log.csv'
log = pd.read_csv(filepath)

In [64]:
log = log.iloc[3:-3]

In [65]:
log.columns

Index(['Tracksys Information', 'FOLDER NUMBER', 'Who Cataloged?',
       'DOCUMENT ID', 'TITLE', 'Document Length',
       'INTERNAL NOTES "Bulk imported" is automatically entered. Add any other notes after it.',
       'Manuscript Type (for Catalog Creation Only) Autograph | Copy | Draft | Fragment | Handwritten | Letterhead | Printed | Signed | Typed | Version',
       'Related Documents (for Catalog Creation Only) Choose General, Enclosure, or Response ',
       'TIFFs Created', 'PDF or IIIF manifest Created', 'UPLOADED TO F.T.P',
       'UVA IIIF IDs', 'UVA IIIF Imported into Drupal',
       'Editor Moved to Storage', 'Notes'],
      dtype='object')

In [66]:
log = log[['DOCUMENT ID', 'TITLE', 'Related Documents (for Catalog Creation Only) Choose General, Enclosure, or Response ', 'UVA IIIF IDs']]
log.columns = ['ID', 'Title', 'Related', 'Images']
log

,ID,Title,Related,Images
3,PJB 9068,"From Julian Bond to Henry Washington, 17 Apr 1974",General: Draft: From Julian Bond to Henry Wash...,https://iiif.lib.virginia.edu/iiif/tsm:3407367...
4,PJB 9069,"Draft: From Julian Bond to Henry Washington, 1...",NaN,https://iiif.lib.virginia.edu/iiif/tsm:3407368...
5,PJB 9070,Envelope: To Julian Bond from Henry Washington...,Response: From Julian Bond to Henry Washington...,https://iiif.lib.virginia.edu/iiif/tsm:3407369...
6,PJB 9071,"From Julian Bond to George Jones, 17 Apr 1974",General: Draft: From Julian Bond to George Jon...,https://iiif.lib.virginia.edu/iiif/tsm:3407371...
7,PJB 9072,"Draft: From Julian Bond to George Jones, 17 Ap...",NaN,https://iiif.lib.virginia.edu/iiif/tsm:3407372...
...,...,...,...,...
141,PJB 9205,"From Julian Bond to Livingstone Johnson, 30 Ap...",NaN,https://iiif.lib.virginia.edu/iiif/tsm:3407521...
142,PJB 9206,"From Julian Bond to C. L. Clifton, 30 Apr 1974",NaN,https://iiif.lib.virginia.edu/iiif/tsm:3407522...
143,PJB 9207,"From Julian Bond to Earnest Pharr, 30 Apr 1974",NaN,https://iiif.lib.virginia.edu/iiif/tsm:3407523...
144,PJB 9208,"From Julian Bond to J. Lowell Ware, 30 Apr 1974",NaN,https://iiif.lib.virginia.edu/iiif/tsm:3407524...


In [67]:
log.ID = log.ID.apply(lambda x: x.strip('PJB '))

In [68]:
replace_map = dict(zip(log.Title, log.ID))
replace_map
replace_map = dict(sorted(replace_map.items(), key=lambda kv: len(kv[0]), reverse=True))

In [69]:
map_file = 'node_field_data.csv'
mapping_df = pd.read_csv(map_file)
mapping_df.shape

(8973, 2)

In [70]:
mapping_df.columns

Index(['nid', 'title'], dtype='object')

In [71]:
replace_map = dict(zip(mapping_df.title, mapping_df.nid))

In [ ]:
# very interesting - the SQL LIMIT function does not apply to export (but that's fine, better more than less)
# will need to export and add to this incrementally, but once the architecture is in place that seems like not such a big deal


In [72]:
def multi_replace(text, replace_map):
    """
    Replace multiple substrings in `text` based on a mapping dict.
    - Keys are replaced with their values
    - Longest keys are prioritized
    - Special regex characters in keys are escaped
    """
    if not isinstance(text, str):
        return text  # return unchanged if not a string
    
    replace_map = {str(k): str(v) for k, v in replace_map.items()}
    
    # Sort keys by length (longest first)
    sorted_keys = sorted(replace_map.keys(), key=len, reverse=True)
    
    # Build regex pattern with escaped keys
    pattern = re.compile("|".join(re.escape(k) for k in sorted_keys))
    
    # Replace using a lambda that looks up the match
    return pattern.sub(lambda m: replace_map[m.group(0)], text)

In [75]:
log['Related'] = log['Related'].apply(lambda x: multi_replace(x, replace_map))

In [89]:
related_export = log[['ID', 'Related']]

In [90]:
def extract_labels(text):
    labels = ["General", "Enclosure", "Response"]
    out = {lbl: None for lbl in labels}
    out["unmatched"] = False   # flag
    
    if not isinstance(text, str):
        return out
    
    consumed_spans = []  # track matched spans
    
    for label in labels:
        # find all occurrences like "General: 1, 2"
        for m in re.finditer(rf"{label}:\s*([\d, ]+)", text):
            consumed_spans.append(m.span())
            numbers = [n.strip() for n in m.group(1).split(",") if n.strip()]
            
            if out[label]:
                out[label] += "|" + "|".join(numbers)
            else:
                out[label] = "|".join(numbers)
    
    # Mark unmatched text if there is leftover after removing matches
    if consumed_spans:
        mask = [" "] * len(text)
        for start, end in consumed_spans:
            mask[start:end] = "_" * (end-start)  # mark matched regions
        leftover = "".join(ch for ch, m in zip(text, mask) if m == " ")
        if leftover.strip():   # anything left that isn’t just spaces/commas
            out["unmatched"] = True
    else:
        # Nothing matched at all → flag whole text as unmatched
        if text.strip():
            out["unmatched"] = True
    
    return out

In [91]:
expanded = related_export["Related"].apply(extract_labels).apply(pd.Series)
related_export = pd.concat([related_export, expanded], axis=1)

In [92]:
related_export = related_export.drop('Related', axis = 1)

In [94]:
related_export.unmatched.value_counts()

unmatched
False    139
True       4
Name: count, dtype: int64

In [96]:
# so it flagged what it should have...
related_feed = related_export[related_export['unmatched'] == False].drop('unmatched', axis = 1)
related_feed.to_csv('related_docs_b17f8_feed.csv', index = False)
needs_manual_entry = related_export[related_export['unmatched'] == True].drop('unmatched', axis = 1)
needs_manual_entry.to_csv('needs_manual_entry_b17f8.csv', index = False)

In [86]:
# Chalking this up as a success assuming we can get the feed importer to work.